# Week 1-2 exploration

Goal: look at the raw postings we collected, get a first sense of the shape of the data, and identify one insight worth writing a LinkedIn post about.

This is NOT a dashboard. This is the lab notebook where I poke at the data and get surprised.

## Load data

Query straight from BigQuery. (If you're running dry-run mode, point `pd.read_json` at your local JSONL instead.)

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project=os.environ['GCP_PROJECT'])

df = client.query('''
    SELECT *
    FROM `raw.jobs`
    WHERE DATE(scraped_at) >= DATE_SUB(CURRENT_DATE(), INTERVAL 14 DAY)
''').to_dataframe()

print(f'Total rows: {len(df):,}')
df.head()

## Shape of the data

Before any insight, understand: what's missing, what's skewed, what looks suspicious.

In [ ]:
# Completeness
pd.DataFrame({
    'non_null_pct': (df.notna().sum() / len(df) * 100).round(1),
    'unique_values': df.nunique(),
})

In [ ]:
# Volume by source and country
df.groupby(['source', 'country']).size().unstack(fill_value=0)

In [ ]:
# Description length distribution — sanity check that scrapers captured full JDs
df['description_len'] = df['description'].str.len()
df.groupby('source')['description_len'].describe()

## First surface-level looks

These aren't answers yet — they're to generate questions.

In [ ]:
# Top companies posting
df['company'].value_counts().head(20)

In [ ]:
# Top locations
df['location'].value_counts().head(20)

In [ ]:
# Most-common words in titles (very rough — proper skill extraction comes in weeks 3-4)
from collections import Counter
import re

words = Counter()
for title in df['title'].dropna():
    for word in re.findall(r'\b[a-zA-ZÀ-ÿ]{3,}\b', title.lower()):
        words[word] += 1

pd.Series(dict(words.most_common(30)))

## Open questions for next week

_Fill this in with things you noticed that you couldn't answer yet. These become next week's work._

- 
- 
- 